# Gold — Economic Dashboard

Joins Silver tables into a single monthly snapshot for the Power BI dashboard.

**Sources:** `silver.exchange_rates`, `silver.central_bank_indicators`, `silver.gdp_indicators`  
**Output:** `gold.economic_dashboard` (Delta table)  
**Grain:** One row per calendar month  

| Column | Description |
|---|---|
| `date` | First day of the month |
| `avg_iskusd` | Monthly average ISK/USD rate |
| `avg_iskeur` | Monthly average ISK/EUR rate |
| `avg_eurusd` | Monthly average EUR/USD rate |
| `avg_omx` | Monthly average OMX Iceland All-Share Index |
| `policy_rate` | End-of-month Central Bank policy rate (%) |
| `cpi` | CPI year-on-year inflation (%) |
| `gdp_yoy_growth` | Quarterly GDP YoY growth (%) mapped to each month |

In [ ]:
df_fx = spark.read.format("delta").table("silver.exchange_rates")
df_cb = spark.read.format("delta").table("silver.central_bank_indicators")
df_gdp = spark.read.format("delta").table("silver.gdp_indicators")

print(f"Exchange rates:  {df_fx.count()} rows")
print(f"Central bank:    {df_cb.count()} rows")
print(f"GDP:             {df_gdp.count()} rows")

In [ ]:
df_fx.createOrReplaceTempView("v_exchange_rates")
df_cb.createOrReplaceTempView("v_central_bank_indicators")
df_gdp.createOrReplaceTempView("v_gdp_indicators")

In [ ]:
df_gold = spark.sql("""
    WITH fx_monthly AS (
        SELECT
            YEAR(date)                    AS year,
            MONTH(date)                   AS month,
            ROUND(AVG(close_iskusd_x), 6) AS avg_iskusd,
            ROUND(AVG(close_iskeur),   6) AS avg_iskeur,
            ROUND(AVG(close_eurusd_x), 6) AS avg_eurusd,
            ROUND(AVG(close_omx),      2) AS avg_omx
        FROM v_exchange_rates
        GROUP BY YEAR(date), MONTH(date)
    ),
    policy AS (
        SELECT year, month, ROUND(value, 4) AS policy_rate
        FROM (
            SELECT
                YEAR(date)  AS year,
                MONTH(date) AS month,
                value,
                ROW_NUMBER() OVER (
                    PARTITION BY YEAR(date), MONTH(date)
                    ORDER BY date DESC
                ) AS rn
            FROM v_central_bank_indicators
            WHERE metric = 'policy_rate'
        )
        WHERE rn = 1
    ),
    cpi AS (
        SELECT
            YEAR(date)   AS year,
            MONTH(date)  AS month,
            value        AS cpi
        FROM v_central_bank_indicators
        WHERE metric = 'cpi'
    ),
    gdp AS (
        SELECT year, q, value AS gdp_yoy_growth
        FROM v_gdp_indicators
    )
    SELECT
        fx.year,
        fx.month,
        TO_DATE(CONCAT_WS('-', fx.year, fx.month, '1'), 'yyyy-M-d') AS date,
        fx.avg_iskusd,
        fx.avg_iskeur,
        fx.avg_eurusd,
        fx.avg_omx,
        p.policy_rate,
        c.cpi,
        g.gdp_yoy_growth,
        CURRENT_TIMESTAMP() AS refreshed_at
    FROM fx_monthly fx
    LEFT JOIN policy p ON fx.year = p.year AND fx.month = p.month
    LEFT JOIN cpi    c ON fx.year = c.year AND fx.month = c.month
    LEFT JOIN gdp    g ON fx.year = g.year
        AND CASE
            WHEN fx.month IN (1,2,3) THEN 1
            WHEN fx.month IN (4,5,6) THEN 2
            WHEN fx.month IN (7,8,9) THEN 3
            ELSE 4
        END = g.q
    ORDER BY fx.year, fx.month
""")

if df_gold.isEmpty():
    raise ValueError("[gold_economic_dashboard] No rows returned - halting.")

df_gold.show(10)

In [ ]:
df_gold.createOrReplaceTempView("v_gold_dashboard")

if not spark.catalog.tableExists("gold.economic_dashboard"):
    df_gold.write.format("delta").saveAsTable("gold.economic_dashboard")
else:
    spark.sql("""
        MERGE INTO gold.economic_dashboard AS target
        USING v_gold_dashboard AS source
        ON target.year = source.year AND target.month = source.month
        WHEN MATCHED THEN UPDATE SET
            target.date           = source.date,
            target.avg_iskusd     = source.avg_iskusd,
            target.avg_iskeur     = source.avg_iskeur,
            target.avg_eurusd     = source.avg_eurusd,
            target.avg_omx        = source.avg_omx,
            target.policy_rate    = source.policy_rate,
            target.cpi            = source.cpi,
            target.gdp_yoy_growth = source.gdp_yoy_growth,
            target.refreshed_at   = source.refreshed_at
        WHEN NOT MATCHED THEN INSERT (
            year, month, date, avg_iskusd, avg_iskeur, avg_eurusd, avg_omx,
            policy_rate, cpi, gdp_yoy_growth, refreshed_at
        )
        VALUES (
            source.year, source.month, source.date, source.avg_iskusd, source.avg_iskeur,
            source.avg_eurusd, source.avg_omx, source.policy_rate, source.cpi,
            source.gdp_yoy_growth, source.refreshed_at
        )
    """)

print(f"Saved to gold.economic_dashboard - {spark.table('gold.economic_dashboard').count()} rows")